In [1]:
# Pour le data management 
import pandas as pd
import numpy as np

# Pour le pré processing
from unidecode import unidecode
import re
from nltk.stem import SnowballStemmer

# Les bigrammes
from collections import Counter
from nltk.util import ngrams

# Pour la vectorisation
from sklearn.feature_extraction.text import TfidfVectorizer 

# Pour la modélisation
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score

from sklearn.naive_bayes import GaussianNB

In [2]:
with open("datasets/article.txt", "r", encoding="utf-8") as file:
    raw_text = file.read().lower()

In [3]:
mots = raw_text.split()
compte_mots = Counter(mots) 
compte_mots

Counter({'de': 39,
         'les': 20,
         'troubles': 14,
         'la': 11,
         'psychiatriques': 10,
         'des': 10,
         'et': 9,
         'à': 8,
         'ont': 7,
         'un': 6,
         'après': 5,
         'covid-19': 5,
         'cette': 5,
         'étude': 5,
         'été': 5,
         'pour': 5,
         'dans': 4,
         'patients': 4,
         'a': 4,
         'prédire': 3,
         'par': 3,
         'facteurs': 3,
         'qui': 3,
         'plus': 3,
         'aiguë': 3,
         'marqueurs': 3,
         'nouveaux': 3,
         "l'apparition": 2,
         'du': 2,
         'épisode': 2,
         'd’une': 2,
         'le': 2,
         'septembre': 2,
         'ayant': 2,
         'phase': 2,
         'psychiatriques,': 2,
         'dépressifs': 2,
         'mesurés': 2,
         'permettent': 2,
         'antécédents': 2,
         'deux': 2,
         'ans': 2,
         '34': 2,
         '489': 2,
         'entre': 2,
         'durée': 2,
      

In [4]:
import nltk
from nltk.corpus import stopwords

# Télécharger la liste des stopwords (une seule fois)
nltk.download('stopwords')

# Récupérer la liste française
stopWords = stopwords.words('french')

print(len(stopWords))
print(stopWords[:20])  # pour voir un aperçu

157
['au', 'aux', 'avec', 'ce', 'ces', 'dans', 'de', 'des', 'du', 'elle', 'en', 'et', 'eux', 'il', 'ils', 'je', 'la', 'le', 'les', 'leur']


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/andreal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
stemmer = SnowballStemmer("french")

In [6]:
# Création d'une fonction pour supprimer les sw
def no_stop_word(string, stopWords):

    """
    Supprime les stop words d'un texte.

    Paramètres
    ----------

    string : chaine de caractère.

    stopWords : liste de mots à exclure. 
    
    ----------
    Sortie : string sans stopWords
    """
    string = ' '.join([word for word in string.split() if word not in stopWords])
    return string
    
    

# Création d'une fonction pour stemmatiser chaque mot d'un text 
def stemmatise_text(text, stemmer):

    """
    Stemmatise un texte : Ramène les mots d'un texte à leur racine (peut créer des mots qui n'existe pas).

    Paramètres
    ----------

    text : Chaine de caractères.

    stemmer : Stemmer de NLTK.
    
    ----------
    Sortie : string qui contient la forme stemmatisée des mots
    """
    string = ' '.join([stemmer.stem(word) for word in text.split()])
    return string
    


In [7]:
mots = [m.lower() for m in mots]

In [8]:
mots = [unidecode(m) for m in mots]

In [9]:
mots = [re.sub(r"\b\d{4}\b", "annee", x) for x in mots]

In [10]:
texte = str(" ").join(mots)

In [11]:
texte = re.sub(r"[^a-z]+"," ",texte) 

In [12]:
texte

'predire l apparition de troubles psychiatriques apres un covid severe les equipes du service de psychiatrie de l hopital bicetre ap hp de la faculte de medecine de l universite paris saclay et de l inserm equipe moods cesp coordonnees par les docteurs matthieu gasnier et romain colle ont cherche a savoir s il etait possible d identifier des facteurs qui favorisent l apparition des troubles psychiatriques apres un episode de covid severe les resultats de cette etude ont fait l objet d une publication parue le septembre annee dans la revue molecular psychiatry plus de millions de personnes dans le monde ont ete touchees par un covid severe ayant entraine des hospitalisations parmi les complications survenues apres la phase aigue de la maladie on denombre des troubles psychiatriques tels que les troubles depressifs et anxieux cette etude visait donc a determiner si des marqueurs cliniques et biologiques mesures durant l hospitalisation pour covid permettent de predire l apparition de nou

In [13]:
texte = no_stop_word(texte, stopWords)

In [14]:
texte

'predire apparition troubles psychiatriques apres covid severe equipes service psychiatrie hopital bicetre ap hp faculte medecine universite paris saclay inserm equipe moods cesp coordonnees docteurs matthieu gasnier romain colle cherche a savoir etait possible identifier facteurs favorisent apparition troubles psychiatriques apres episode covid severe resultats cette etude fait objet publication parue septembre annee revue molecular psychiatry plus millions personnes monde ete touchees covid severe entraine hospitalisations parmi complications survenues apres phase aigue maladie denombre troubles psychiatriques tels troubles depressifs anxieux cette etude visait donc a determiner si marqueurs cliniques biologiques mesures durant hospitalisation covid permettent predire apparition nouveaux troubles psychiatriques chez patients sans antecedents psychiatriques suivis jusqu a deux ans apres phase aigue maladie cette etude cohorte prospective a ete menee a partir entrepot donnees sante eds

In [15]:
texte_stem = stemmatise_text(texte, stemmer)

In [16]:
texte_stem

'predir apparit troubl psychiatr apre covid sever equip servic psychiatr hopital bicetr ap hp facult medecin universit paris saclay inserm equip mood cesp coordonne docteur matthieu gasni romain coll cherch a savoir etait possibl identifi facteur favorisent apparit troubl psychiatr apre episod covid sever resultat cet etud fait objet publiqu paru septembr anne revu molecular psychiatry plus million person mond ete touche covid sever entrain hospitalis parm compliqu survenu apre phas aigu malad denombr troubl psychiatr tel troubl depress anxieux cet etud vis donc a determin si marqueur cliniqu biolog mesur dur hospitalis covid permettent predir apparit nouveau troubl psychiatr chez patient san antecedent psychiatr suiv jusqu a deux an apre phas aigu malad cet etud cohort prospect a ete mene a part entrepot donne sant ed ap hp comport dossi patient age plus an hospitalis covid hopital parisien entre janvi anne septembr anne etud a exclu patient antecedent psychiatr isol cas nouveau troub

In [ ]:
# tokenisation et voir si on peut faire un lemming du texte puis faire le wordcloud